### ***Python Code***

In [ ]:
from scipy.optimize import linprog

# Coefficients of the objective function (negative because linprog minimizes)
c = [-0.12, -0.09]

# Coefficients for inequality constraints
A = [[1, 1], [6, 4]]
b = [50000, 240000]

# Bounds for the variables
x1_bounds = (0, 35000)
x2_bounds = (0, 50000)

# Solve the linear program
result = linprog(c, A_ub=A, b_ub=b, bounds=[x1_bounds, x2_bounds], method='highs')

# Output the results
if result.success:
    print(f"Optimal investment in Internet fund: ${result.x[0]:.2f}")
    print(f"Optimal investment in Blue Chip fund: ${result.x[1]:.2f}")
    print(f"Maximum return: ${-result.fun:.2f}")
else:
    print("Optimization failed.")


Optimal investment in Internet fund: $20000.00
Optimal investment in Blue Chip fund: $30000.00
Maximum return: $5100.00


### **AMPL Code**

In [ ]:
# Install dependencies
!pip install -q amplpy ampltools

# Google Colab & AMPL integration
MODULES, LICENSE_UUID = ["coin", 'gurobi', "cplex", "highs", "gokestrel"], "42fc7eb6-69aa-445d-b655-3ad24d836541"
from amplpy import tools
from ampltools import cloud_platform_name, ampl_notebook, register_magics

# instantiate AMPL object and register magics
if cloud_platform_name() is None:
    ampl = AMPL() # Use local installation of AMPL
else:
    ampl = tools.ampl_notebook(modules=MODULES, license_uuid=LICENSE_UUID, g=globals())

register_magics(ampl_object=ampl)


Licensed to Bundle #6741.7193 expiring 20241231: INFO 645 Prescriptive Analytics, Prof. Paul Brooks, Virginia Commonwealth University.


In [ ]:
# Define model
ampl.eval ('''

reset;

# Define sets
set FUNDS;

# Define parameters
param total_budget;
param max_investment_internet;
param total_risk;
param return_rate{FUNDS};
param risk_rate{FUNDS};

# Define decision variables
var x{f in FUNDS} >= 0; # investment amount for each fund

# Objective function: Maximize total return
maximize total_return: sum {f in FUNDS} return_rate[f] * x[f];

# Constraints
subject to budget_constraint: sum {f in FUNDS} x[f] <= total_budget;

subject to internet_limit: x['Internet'] <= max_investment_internet;

subject to risk_constraint: sum {f in FUNDS} risk_rate[f] * x[f] <= total_risk;

''')

# Define data and provide to AMPL model
funds = ['Internet', 'BlueChip']
return_rates = {'Internet': 0.12, 'BlueChip': 0.09}
risk_rates = {'Internet': 6, 'BlueChip': 4}

ampl.set['FUNDS'] = funds
ampl.param['total_budget'] = 50000
ampl.param['max_investment_internet'] = 35000
ampl.param['total_risk'] = 240000
ampl.param['return_rate'] = return_rates
ampl.param['risk_rate'] = risk_rates

In [ ]:
# Use ampl.expand to confirm AMPL model syntax is working
ampl.eval('expand;')

# Set solver option and solve
ampl.setOption('solver', 'cbc')

# Print results
ampl.solve()

maximize total_return:
	0.12*x['Internet'] + 0.09*x['BlueChip'];

subject to budget_constraint:
	x['Internet'] + x['BlueChip'] <= 50000;

subject to internet_limit:
	x['Internet'] <= 35000;

subject to risk_constraint:
	6*x['Internet'] + 4*x['BlueChip'] <= 240000;

cbc 2.10.10: cbc 2.10.10: optimal solution; objective 5100
0 simplex iterations


In [ ]:
obj = ampl.get_objective('total_return')
print("\nTotal return is: ", obj.get().value(), "\n")
print("Investment in each fund:")
ampl.display('x');


Total return is:  5100.0 

Investment in each fund:
x [*] :=
BlueChip  30000
Internet  20000
;

